In [4]:
%pip install accelerate peft bitsandbytes transformers trl==0.16.0 scipy numpy==1.26.4 liger-kernel==0.5.5

  Using cached bitsandbytes-0.45.4-py3-none-manylinux_2_24_x86_64.whl.metadata (5.0 kB)
  Using cached trl-0.16.0-py3-none-any.whl.metadata (12 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached liger_kernel-0.5.5-py3-none-any.whl.metadata (22 kB)
  Using cached datasets-3.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.5.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2024.12.0-py3-none-any.whl.metadata (11 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached

In [1]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: write).
The token `Upload` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Upload`


In [ ]:
!trl sft \
    --model_name_or_path gpt2 \
    --dataset_name bigscience-data/roots_vi_wikiquote \
    --max_length 1024 \
    --num_train_epochs 1 \
    --learning_rate 1e-4 \
    --lr_scheduler_type cosine \
    --packing \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --gradient_checkpointing \
    --save_steps 100 \
    --save_total_limit 2 \
    --warmup_ratio 0.05 \
    --use_liger_kernel \
    --logging_steps 1 \
    --output_dir gpt2-finetuned \
    --fp16

2025-03-31 14:34:21.043949: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743431661.318950    2744 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743431661.391078    2744 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-31 14:34:21.973790: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-31 14:34:37.647177: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: 

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTConfig, SFTTrainer

# Load the dataset
dataset = load_dataset(
    'bigscience-data/roots_vi_wikiquote',
    split="train"
)

# Load the model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    'gpt2',
    quantization_config=None,
    device_map='auto',
    # attn_implementation="flash_attention_2"
)
tokenizer = AutoTokenizer.from_pretrained(
    'gpt2',
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

# Define training arguments
training_arguments = SFTConfig(
    max_length=1024, # Depend on model max context-length
    use_liger=True,
    output_dir='ckpt',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    optim='adamw_torch',
    save_steps=1000,
    save_total_limit=100,
    logging_steps=100,
    learning_rate=1e-4,
    weight_decay=0.001,
    fp16=True,  # If GPU >= A100, set this to False
    bf16=False,  # If GPU >= A100, set this to True
    max_grad_norm=1.0,
    warmup_ratio=0.05,
    group_by_length=True,
    lr_scheduler_type='cosine',
    dataloader_num_workers=2,  # Number of GPUs
    push_to_hub=False,
    report_to="none",
    # load_best_model_at_end=True # If you have a validation dataset
)

# Create the trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=None,
    args=training_arguments,
)

# Train model
trainer.train(resume_from_checkpoint=False)

# Save trained model
trainer.model.save_pretrained('gpt2-finetuned')


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


KeyboardInterrupt: 